# Exploratory Data Analysis (EDA)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import SparkFiles
from pyspark.ml.feature import HashingTF, MinHashLSH
import fasttext
import unicodedata
import time
import os
from pathlib import Path
from urllib.request import urlretrieve

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 10g pyspark-shell"
)

spark = (
    SparkSession.builder
    .appName("exploration")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

heap_gb = (
    spark._jvm.java.lang.Runtime
    .getRuntime()
    .maxMemory() / 1024**3
)

print(f"JVM maximum heap: {heap_gb:.2f} GB")

Spark version: 4.2.0
Spark master: local[*]
JVM maximum heap: 10.00 GB


# Read in the dataset

In [2]:
df = spark.read.parquet(
    "aztc_full_localdoc/train-00000-of-00018.parquet"
)

df.printSchema()
df.show(5, truncate=100)

root
 |-- text: string (nullable = true)
 |-- source: string (nullable = true)

+----------------------------------------------------------------------------------------------------+------+
|                                                                                                text|source|
+----------------------------------------------------------------------------------------------------+------+
|Aqil Abbas: "Mən ədəbiyyat adamıyam, AMMA JURNALİSTİKA DA ƏDƏBİYYATDAN KƏNARDA DEYİL". Gülüş günü...|   ANL|
|Geriyə qayıtmaq imkanım olsaydı, buraxdığım səhvləri təkrarlamazdım". AZƏRBAYCAN MİLLİ KOMANDASIN...|   ANL|
|Elçinin "Qatil"i Moskvada Türkiyə və Azərbaycanı təmsil edib. Çağdaş Türkiyə teatr sənətinin bir ...|   ANL|
|Nurəddin Qənbərin yeni kitabı nəşr edilib. "Bakı-Şirvannəşr"də yazıçı-cərrah Nurəddin Qənbərin "Y...|   ANL|
|Borçalının keçmişinə və gələcəyinə daha bir baxış. Kitabda Gürcüstanın Borçalı və ona bitişik böl...|   ANL|
+---------------------------------------

# Check for Source distribution

In [11]:
df.groupBy("source").count().orderBy(F.desc("count")).show(100, truncate=False)

+--------+------+
|source  |count |
+--------+------+
|ANL     |240655|
|apasport|4031  |
+--------+------+



### Instruction: Remove all sources except "ANL"

# Check for null and empty texts

In [3]:
df.agg(
    F.count("*").alias("documents"),
    F.count_if(F.col("text").isNull()).alias("null_text"),
    F.count_if(F.col("text") == "").alias("empty_text"),
).show()

+---------+---------+----------+
|documents|null_text|empty_text|
+---------+---------+----------+
|   244686|        0|         0|
+---------+---------+----------+



### Instruction: If any null or empty text found remove those rows entirely.

# Check whitespace-only documents

In [4]:
df.agg(
    F.count_if(F.col("text").rlike(r"(?U)^\s+$")).alias("whitespace_only")
).show()

+---------------+
|whitespace_only|
+---------------+
|              0|
+---------------+



### Instruction: If any whitespace_only found then remove those rows entirely.

# Check character distribution

In [8]:
# "Other" = anything outside Latin script, numbers, punctuation and common ASCII whitespace.
patterns = {
    "latin_script": r"\p{IsLatin}",
    "numbers": r"\p{N}",
    "punctuation": r"\p{P}",
    "common_whitespace": r"[ \t\r\n\f]",
    "other": r"[^\p{IsLatin}\p{N}\p{P} \t\r\n\f]",
}
df.agg(*(
    F.sum(F.length("text") - F.length(F.regexp_replace("text", p, ""))).alias(k)
    for k, p in patterns.items()
)).show(truncate=False)

+------------+-------+-----------+-----------------+-----+
|latin_script|numbers|punctuation|common_whitespace|other|
+------------+-------+-----------+-----------------+-----+
|1114469817  |4668754|31732330   |170164689        |79483|
+------------+-------+-----------+-----------------+-----+



In [21]:
# Character frequency matrix for the "other" category
other_character_distribution = (
    df.select(F.regexp_replace(
        "text", r"[\p{IsLatin}\p{N}\p{P} \t\r\n\f]", ""
    ).alias("chars"))
    .select(F.explode(F.expr(
        "transform(sequence(1, length(chars)), i -> substring(chars, i, 1))"
    )).alias("char"))
    .where(F.col("char") != "")
    .groupBy("char")
    .count()
    .orderBy(F.desc("count"))
)

rows = other_character_distribution.collect()

print(f"{'Character':<12} {'Unicode':<12} {'Name':<45} {'Count':>12}")
print("-" * 85)

for r in rows:
    print(
        f"{repr(r.char):<12} "
        f"U+{ord(r.char):04X}      "
        f"{unicodedata.name(r.char, 'UNASSIGNED'):<45} "
        f"{r['count']:>12,}"
    )

Character    Unicode      Name                                                 Count
-------------------------------------------------------------------------------------
'а'          U+0430      CYRILLIC SMALL LETTER A                             21,231
'х'          U+0445      CYRILLIC SMALL LETTER HA                            14,104
'ә'          U+04D9      CYRILLIC SMALL LETTER SCHWA                          7,758
'о'          U+043E      CYRILLIC SMALL LETTER O                              6,905
'е'          U+0435      CYRILLIC SMALL LETTER IE                             6,793
'¬'          U+00AC      NOT SIGN                                             3,354
'к'          U+043A      CYRILLIC SMALL LETTER KA                             1,985
'№'          U+2116      NUMERO SIGN                                          1,705
'+'          U+002B      PLUS SIGN                                            1,430
'ю'          U+044E      CYRILLIC SMALL LETTER YU                        

### Instruction: Remove the text rows that contain any of the "other" characters above.

In [32]:
cyrillic_texts = df.filter(
    F.length(F.regexp_replace("text", r"[^\p{IsCyrillic}]", "")) >= 1
)

count = cyrillic_texts.count()
total = df.count()

print(f"Documents with Cyrillic: {count:,} / {total:,} ({count / total:.2%})")

for i, row in enumerate(cyrillic_texts.select("source", "text").take(5), 1):
    matches = list(__import__("re").finditer(r"[\u0400-\u052F]", row.text))

    print(f"\n--- Sample {i} | Source: {row.source} ---")
    print("Cyrillic characters:", "".join(m.group() for m in matches[:30]))

    for match in matches[:5]:
        start, end = match.span()
        print(repr(row.text[max(0, start - 50):min(len(row.text), end + 50)]))

Documents with Cyrillic: 1,445 / 244,686 (0.59%)

--- Sample 1 | Source: ANL ---
Cyrillic characters: цццц
'iqəti tapmaq hüquqda, məhkəmədə, elmdə olduqca mühцmdцr. Sənət həqiqətdən daha çox yalan цstə qurulur'
'ti tapmaq hüquqda, məhkəmədə, elmdə olduqca mühцmdцr. Sənət həqiqətdən daha çox yalan цstə qurulur. S'
'olduqca mühцmdцr. Sənət həqiqətdən daha çox yalan цstə qurulur. Sənətin həqiqəti həyatın həqiqətindən'
' ithafda adının baş hərfi keçən A-nın kimliyinin mцəmmalı qalmasıdır. Allah bizi yaxşı qurtarıb).'

--- Sample 2 | Source: ANL ---
Cyrillic characters: ееееееееееееееееееееееееееееее
'– bu səs toya üçcə gün qalmış bəy еvindən çıxıb astananın bеş addımlığınadək yayılırd'
'toya üçcə gün qalmış bəy еvindən çıxıb astananın bеş addımlığınadək yayılırdı. Dəvətnamələr paylanmış'
'mur sözü bəlkə də vəziyyəti tam dəqiqliyi ilə əks еtdirmir. Bəlkə də təzə bəy, yəni еvin ilk bеşiyi V'
'qiqliyi ilə əks еtdirmir. Bəlkə də təzə bəy, yəni еvin ilk bеşiyi Vəli canından kеçib, qıçına-topuğu

### Instruction: Remove all text rows that contains Cyrillic

# Check document length distribution

In [10]:
df.select(F.length("text").alias("characters")).summary(
    "count", "mean", "min", "25%", "50%", "75%", "90%", "99%", "max"
).show()

+-------+-----------------+
|summary|       characters|
+-------+-----------------+
|  count|           244686|
|   mean|5399.226245065104|
|    min|               20|
|    25%|             1487|
|    50%|             3793|
|    75%|             7850|
|    90%|            11273|
|    99%|            24615|
|    max|           171892|
+-------+-----------------+



### Instruction: Remove all text rows that are less than 500 characters

# Analyze language distribution (important)

In [ ]:
%pip install --only-binary=:all: fasttext-numpy2==0.10.4

In [5]:
model_path = Path("/tmp/lid.176.bin")

if not model_path.exists():
    urlretrieve(
        "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin",
        model_path
    )

Model ready: /tmp/lid.176.bin


In [36]:
model = fasttext.load_model(str(model_path))

samples = [
    "'Afaq Sultanova yeni çapdan  çıxmış “Души осенние напевы” (“Qəlbin payız nəğmələri”)  adlı şeirlər top'",
    "Dünən axşam dostumla birlikdə kinoya getdim və orada çox maraqlı bir film izlədim.",
    "egionun geosiyasi prosesləri  barədə fikirlərini bцlьєьb: “Dünya düzəninin yenidən qurulduğu bir dövr'",
    "Bu gün hava çok gözəl."
]

print('If this outputs classification then FastText is installed correctly\n')
for text in samples:
    labels, scores = model.predict(text, k=2)
    print(text)
    print([
        (label.removeprefix("__label__"), round(float(score), 3))
        for label, score in zip(labels, scores)
    ])

def test_worker(_):
    import fasttext
    from pyspark import SparkFiles

    worker_model = fasttext.load_model(SparkFiles.get("lid.176.bin"))
    return worker_model.predict("Azərbaycanın paytaxtı Bakı şəhəridir.")[0][0]


print(
    "Worker test:",
    spark.sparkContext.parallelize([0], 1).map(test_worker).collect()
)

If this outputs classification then FastText is installed correctly

'Afaq Sultanova yeni çapdan  çıxmış “Души осенние напевы” (“Qəlbin payız nəğmələri”)  adlı şeirlər top'
[('az', 0.916), ('tr', 0.059)]
Dünən axşam dostumla birlikdə kinoya getdim və orada çox maraqlı bir film izlədim.
[('az', 0.999), ('hy', 0.0)]
egionun geosiyasi prosesləri  barədə fikirlərini bцlьєьb: “Dünya düzəninin yenidən qurulduğu bir dövr'
[('az', 0.999), ('tr', 0.001)]
Bu gün hava çok gözəl.
[('tr', 0.996), ('az', 0.004)]
Worker test: ['__label__az']


In [19]:
def detect_partition(rows):
    classifier = fasttext.load_model("/tmp/lid.176.bin")

    for row in rows:
        text = row.text
        language = (
            classifier.predict(" ".join(text.split()))[0][0].removeprefix("__label__")
            if text and text.strip() else "undetermined"
        )
        yield row.source, text, language

language_results = df.select("source", "text").rdd.mapPartitions(detect_partition).cache()

counts = language_results.map(lambda row: row[2]).countByValue()
total = sum(counts.values())

print(f"Total documents: {total:,}\n")
print(f"{'Language':<15} {'Documents':>12} {'Percentage':>12}")

for lang, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"{lang:<15} {count:>12,} {count / total:>11.2%}")

Total documents: 244,686

Language           Documents   Percentage
az                   244,669      99.99%
tr                        17       0.01%


In [39]:
turkish_samples = (
    language_results
    .filter(lambda row: row[2] == "tr")
    .take(5)
)

for i, (source, text, language) in enumerate(turkish_samples, 1):
    print(f"\n--- Turkish sample {i} | Source: {source} ---")
    print(text[:1000])


--- Turkish sample 1 | Source: ANL ---
ABBAS ABDULLA’NIN ŞİİRLER KİTABI TÜRKİYE’DE. Ölemmedim amma. Ve Hacaloğlu’dan iki şiir. Amma minalanmış torpağam babam. Tofiq ABDİN.

--- Turkish sample 2 | Source: ANL ---
GÜNEŞİ İÇENLERİN TÜRKÜSÜ.

--- Turkish sample 3 | Source: ANL ---
Çok küçük bir yalanı.

--- Turkish sample 4 | Source: ANL ---
Göylə işiniz  olmasın. Kəhkəşanlı bağçadır o, Əl çatmayan  taxçadır o. Göylə işiniz olmasın. Ümidlər gözü ordadır, Kitablar sözü ordadır. Allahın özü ordadır, Göylə işiniz olmasın.

--- Turkish sample 5 | Source: ANL ---
Nə olar, başqasının olanda, bir insan yavrusunu.


### Instruction: Remove all text rows that are classified other than Azerbaijani

# Check for repeated boilerplate (Navigation menus, copyright notices and etc)

In [42]:
boilerplate_candidates = df.select("text").where(F.col("text").isNotNull()).dropDuplicates()

parts = boilerplate_candidates.select(F.posexplode(F.array(
    F.substring("text", 1, 120),
    F.expr("right(text, 120)")
)).alias("position", "fragment"))

results = (
    parts.where(F.length("fragment") == 120)
    .groupBy("position", "fragment")
    .count()
    .where(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .limit(5)
    .collect()
)

for r in results:
    print(f"\n{'Ending' if r.position else 'Beginning'} | Count: {r['count']:,}")
    print(r.fragment)


Ending | Count: 1,042
edib Azərbaycan Respublikasının Nazirlər Kabinetinə məlumat versin. İlham ƏLİYEV, Azərbaycan Respublikasının Prezidenti.

Ending | Count: 765
. Azərbaycan Respublikası Prezidenti yanında KİV-in İnkişafına Dövlət Dəstəyi Fondunun maliyyə dəstəyilə çap edilmişdir.

Ending | Count: 528
Nazirlər Kabineti bu sərəncamdan irəli gələn məsələləri həll etsin. İlham ƏLİYEV, Azərbaycan Respublikasının Prezidenti.

Ending | Count: 314
rədə Azərbaycan Respublikasının Nazirlər Kabinetinə məlumat versin. İlham ƏLİYEV, Azərbaycan Respublikasının Prezidenti.

Ending | Count: 187
unda dəyişiklik edilməsi barədə. Azərbaycan Respublikasının Qanunu. İlham ƏLİYEV, Azərbaycan Respublikasının Prezidenti.


### Instuction: Keep the boilperlates(they are sign-offs) as is.

# Count exact duplicates

In [43]:
# Total number of documents
total = df.count()

# Hash text only (ignore source)
hashed_df = df.select(
    F.sha2(F.col("text"), 256).alias("text_hash")
)

# Count unique text hashes
unique = hashed_df.distinct().count()

# Redundant documents
duplicates = total - unique

print(f"Total documents:     {total:,}")
print(f"Unique texts:        {unique:,}")
print(f"Duplicate documents: {duplicates:,}")
print(f"Duplicate rate:      {duplicates / total * 100:.2f}%" if total else "Duplicate rate: N/A")

# JVM heap usage after completion
runtime = spark._jvm.java.lang.Runtime.getRuntime()

gb = 1024 ** 3
heap_used = runtime.totalMemory() - runtime.freeMemory()
heap_max = runtime.maxMemory()

print()
print(f"JVM heap used:       {heap_used / gb:.2f} GB")
print(f"JVM heap maximum:    {heap_max / gb:.2f} GB")
print(f"JVM heap utilization: {heap_used / heap_max * 100:.2f}%")

# Docker container memory (cgroup v2, if available)
try:
    with open("/sys/fs/cgroup/memory.current") as f:
        memory = int(f.read().strip())

    print(f"Container memory:    {memory / gb:.2f} GB")
except (OSError, ValueError):
    print("Container memory:    unavailable")

Total documents:     244,686
Unique texts:        228,967
Duplicate documents: 15,719
Duplicate rate:      6.42%

JVM heap used:       1.41 GB
JVM heap maximum:    10.00 GB
JVM heap utilization: 14.11%
Container memory:    7.41 GB


### Instuction: Remove all exact duplicates

# Count near duplicates

In [44]:
from pyspark.sql import functions as F


BASE = "/home/jovyan/work/near_duplicate_data_eda"
FEATURE_PATH = f"{BASE}/near_duplicate_features"
PAIRS_PATH = f"{BASE}/near_duplicate_pairs"

# Remove exact duplicates and normalize text
docs = (
    df.select("text")
    .filter(F.col("text").isNotNull())
    .dropDuplicates(["text"])
    .withColumn(
        "words",
        F.split(
            F.trim(F.regexp_replace(
                F.lower("text"),
                r"[^\p{L}\p{N}]+",
                " "
            )),
            r"\s+"
        )
    )
    .filter(F.size("words") >= 10)
)

## Generate unique 5-word shingles (5-grams)
docs = docs.withColumn(
    "shingles",
    F.expr("""
        transform(
            sequence(1, size(words) - 4),
            i -> array_join(slice(words, i, 5), ' ')
        )
    """)
)

# Convert shingles to binary feature vectors
hashing_tf = HashingTF(
    inputCol="shingles",
    outputCol="features",
    numFeatures=1 << 20,
    binary=True
)

features = (
    hashing_tf.transform(docs)
    .select("text", "features")
    .withColumn("id", F.monotonically_increasing_id())
)

# Persist IDs, texts, and features together
features.write.mode("overwrite").parquet(FEATURE_PATH)
features = spark.read.parquet(FEATURE_PATH)

print(f"Documents prepared: {features.count():,}")

Documents prepared: 228,855


In [45]:
start = time.perf_counter()

features = spark.read.parquet(FEATURE_PATH)

# Keep only IDs and feature vectors
vectors = features.select("id", "features")

lsh = MinHashLSH(
    inputCol="features",
    outputCol="hashes",
    numHashTables=3,
    seed=42
)

model = lsh.fit(vectors)

pairs = (
    model.approxSimilarityJoin(
        vectors,
        vectors,
        0.10,
        distCol="distance"
    )
    .filter(F.col("datasetA.id") < F.col("datasetB.id"))
    .select(
        F.col("datasetA.id").alias("id_a"),
        F.col("datasetB.id").alias("id_b"),
        (1 - F.col("distance")).alias("similarity")
    )
)

# Persist near-duplicate pairs
pairs.write.mode("overwrite").parquet(PAIRS_PATH)
pairs = spark.read.parquet(PAIRS_PATH)

print(f"Near-duplicate pairs: {pairs.count():,}")
print(f"Execution time: {time.perf_counter() - start:.2f} seconds")


Near-duplicate pairs: 4,446
Execution time: 332.88 seconds


In [46]:
from pyspark.sql import functions as F

# Load saved results
texts = spark.read.parquet(FEATURE_PATH).select("id", "text")
pairs = spark.read.parquet(PAIRS_PATH)

# Randomly select 5 pairs
sample_pairs = (
    pairs
    .orderBy(F.rand(seed=42))
    .limit(5)
)

# Retrieve original texts only for selected pairs
matches = (
    sample_pairs
    .join(
        texts.select(
            F.col("id").alias("id_a"),
            F.col("text").alias("text_a")
        ),
        "id_a"
    )
    .join(
        texts.select(
            F.col("id").alias("id_b"),
            F.col("text").alias("text_b")
        ),
        "id_b"
    )
    .collect()
)

# Display sampled pairs
for i, row in enumerate(matches, 1):
    print(f"\n{'=' * 80}")
    print(f"PAIR {i} | Similarity: {row.similarity:.2%}")
    print("=" * 80)

    print("\nDOCUMENT A:\n", row.text_a[:2000])
    print("\nDOCUMENT B:\n", row.text_b[:2000])


PAIR 1 | Similarity: 100.00%

DOCUMENT A:
 “Normativ hüquqi aktlar haqqında” Azərbaycan  Respublikasının Konstitusiya Qanununda dəyişiklik edilməsi barədə. Azərbaycan Respublikasının Konstitusiya Qanunu. İlham ƏLİYEV, Azərbaycan Respublikasının Prezidenti.

DOCUMENT B:
 "Normativ hüquqi aktlar haqqında" Azərbaycan  Respublikasının Konstitusiya Qanununda dəyişiklik edilməsi barədə. Azərbaycan Respublikasının Konstitusiya Qanunu. İlham ƏLİYEV, Azərbaycan Respublikasının Prezidenti.

PAIR 2 | Similarity: 97.36%

DOCUMENT A:
 Əhməd İSMAYILOV: “Məqsədimiz  Azərbaycan mediasını güclü  görməkdir”. Azərbaycan Respublikasının Medianın  İnkişafı Agentliyinin (MEDİA)  icraçı direktoru Əhməd  İsmayılovun AZƏRTAC-a müsahibəsi. - Əhməd müəllim, Azərbaycan  mediasının müasir vəziyyəti  və problemləri ilə bağlı ictimai fikrin müxtəlif səviyyələrində  daim müzakirələr gedir  və Milli Mətbuat Günü  yaxınlaşdıqca bu istiqamətdə  fikirlər daha çox  eşidilməkdədir. Azərbaycan mediasının perspektiv inkişaf

### Instruction: Remove all near duplicates (90% similarity cutoff)